In [2]:
!pip install pandas numpy scikit-learn transformers datasets

In [3]:
import pandas as pd
import numpy as np
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
from google.colab import files
uploaded = files.upload()

Saving customer_support_tickets.csv to customer_support_tickets.csv


In [6]:
df = pd.read_csv("/content/drive/MyDrive/customer_support_tickets.csv")

# Combine text
df['text'] = df['Ticket Subject'] + " " + df['Ticket Description']

# Keep only needed columns
df = df[['text', 'Ticket Type']]

df.dropna(inplace=True)

# Reduce size for speed
df = df.sample(300, random_state=42)

df.head()

,text,Ticket Type
4830,Product setup I'm having an issue with the {pr...,Refund request
7075,Battery life I'm having trouble connecting my ...,Product inquiry
4715,Refund request I'm having an issue with the {p...,Billing inquiry
2022,Peripheral compatibility I'm having an issue w...,Billing inquiry
676,Peripheral compatibility I'm having an issue w...,Refund request


In [7]:
print(df.columns)
df.head()

Index(['text', 'Ticket Type'], dtype='object')


,text,Ticket Type
4830,Product setup I'm having an issue with the {pr...,Refund request
7075,Battery life I'm having trouble connecting my ...,Product inquiry
4715,Refund request I'm having an issue with the {p...,Billing inquiry
2022,Peripheral compatibility I'm having an issue w...,Billing inquiry
676,Peripheral compatibility I'm having an issue w...,Refund request


In [9]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['Ticket Type'])

labels = df['Ticket Type'].unique().tolist()
print(labels)

['Refund request', 'Product inquiry', 'Billing inquiry', 'Cancellation request', 'Technical issue']


In [10]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

def zero_shot_predict(text):
    result = classifier(text, labels, multi_label=True)
    return result['labels'][:3]

# Test
print("Zero-shot prediction:")
print(zero_shot_predict(df['text'].iloc[0]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Zero-shot prediction:
['Product inquiry', 'Technical issue', 'Refund request']


In [11]:
df['zero_shot_tags'] = df['text'].head(20).apply(zero_shot_predict)
df[['text', 'zero_shot_tags']].head()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,text,zero_shot_tags
4830,Product setup I'm having an issue with the {pr...,"[Product inquiry, Technical issue, Refund requ..."
7075,Battery life I'm having trouble connecting my ...,"[Product inquiry, Technical issue, Billing inq..."
4715,Refund request I'm having an issue with the {p...,"[Refund request, Product inquiry, Technical is..."
2022,Peripheral compatibility I'm having an issue w...,"[Technical issue, Product inquiry, Billing inq..."
676,Peripheral compatibility I'm having an issue w...,"[Technical issue, Product inquiry, Refund requ..."


In [12]:
def few_shot_prompt(ticket):
    prompt = f"""
You are a support ticket classifier.

Example 1:
Ticket: My laptop is overheating
Tags: Technical Issue

Example 2:
Ticket: I was charged twice
Tags: Billing Inquiry

Example 3:
Ticket: Tell me about product features
Tags: Product Inquiry

Now classify:
Ticket: {ticket}

Return top 3 tags.
"""
    return prompt

print(few_shot_prompt(df['text'].iloc[0]))


You are a support ticket classifier.

Example 1:
Ticket: My laptop is overheating
Tags: Technical Issue

Example 2:
Ticket: I was charged twice
Tags: Billing Inquiry

Example 3:
Ticket: Tell me about product features
Tags: Product Inquiry

Now classify:
Ticket: Product setup I'm having an issue with the {product_purchased}. Please assist. I'm using xda-developer for something different. If there are issues with the {product_purchased} it's likely you are not using the I've tried clearing the cache and data for the {product_purchased} app, but the issue persists.

Return top 3 tags.



In [13]:
dataset = Dataset.from_pandas(df[['text', 'label']])

In [14]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(
        example['text'],
        padding='max_length',
        truncation=True
    )

dataset = dataset.map(tokenize)
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [15]:
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset['train']
test_dataset = dataset['test']

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    "prajjwal1/bert-tiny",
    num_labels=len(labels)
)

model.to(device)

config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect i

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,), eps=1e-12, e

In [42]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=20,
    per_device_eval_batch_size=16,
    num_train_epochs=100,
    logging_steps=20,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

Step,Training Loss
20,0.000569
40,0.000497
60,0.000501
80,0.000451
100,0.016877
120,0.021351
140,0.000610
160,0.001028
180,0.000321
200,0.000301


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.0016977543657412753, metrics={'train_runtime': 37.5718, 'train_samples_per_second': 638.777, 'train_steps_per_second': 31.939, 'total_flos': 30520221696000.0, 'train_loss': 0.0016977543657412753, 'epoch': 100.0})

In [43]:
def predict_top3(text):
    # Clean text
    text = text.replace("{product_purchased}", "")

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    #  Move inputs to GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    # Move back to CPU
    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    top_3_idx = np.argsort(probs)[-3:]

    return [le.inverse_transform([i])[0] for i in top_3_idx[::-1]]

In [44]:
sample_text = df['text'].iloc[5]

print("Ticket:")
print(sample_text)

print("\nPredicted Tags:")
print(predict_top3(sample_text))

Ticket:
Refund request I've encountered a data loss issue with my {product_purchased}. All the files and documents seem to have disappeared. Can you guide me on how to retrieve them?


I cannot verify this information though. It doesn't look very I've noticed that the issue occurs consistently when I use a specific feature or application on my {product_purchased}.

Predicted Tags:
['Billing inquiry', 'Technical issue', 'Product inquiry']


In [45]:
y_true = []
y_pred = []

for text, label in zip(df['text'][:100], df['label'][:100]):
    pred = predict_top3(text)[0]  # top-1 prediction

    y_true.append(label)
    y_pred.append(le.transform([pred])[0])

print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred, average='weighted'))

Accuracy: 0.8
F1 Score: 0.7995527728085868


In [46]:
for i in range(5):
    print("\n---------------------------")
    print("Ticket:", df['text'].iloc[i])
    print("Predicted Tags:", predict_top3(df['text'].iloc[i]))


---------------------------
Ticket: Product setup I'm having an issue with the {product_purchased}. Please assist. I'm using xda-developer for something different. If there are issues with the {product_purchased} it's likely you are not using the I've tried clearing the cache and data for the {product_purchased} app, but the issue persists.
Predicted Tags: ['Refund request', 'Product inquiry', 'Cancellation request']

---------------------------
Ticket: Battery life I'm having trouble connecting my {product_purchased} to my home Wi-Fi network. It doesn't detect any networks, although other devices are connecting fine. What can be done to resolve this issue? I will refer to this issue I've checked for any available software updates for my {product_purchased}, but there are none.
Predicted Tags: ['Technical issue', 'Billing inquiry', 'Cancellation request']

---------------------------
Ticket: Refund request I'm having an issue with the {product_purchased}. Please assist.

Please give c

In [47]:
save_path = "/content/drive/MyDrive/ticket_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/ticket_model/tokenizer_config.json',
 '/content/drive/MyDrive/ticket_model/tokenizer.json')